# Pipeline сбора и подготовки данных League of Legends

## Описание

Данный ноутбук реализует полный процесс сбора и подготовки данных для анализа рейтинговых матчей League of Legends.

### Последовательность выполнения

1. Загрузка Riot API ключа из файла `.env`.

2. Получение списка топ-300 игроков высших рейтинговых лиг:

   * Challenger;
   * Grandmaster;
   * Master.

3. Сбор данных для двух игровых регионов:

   * EUW;
   * NA.

4. Получение информации об игроках:

   * PUUID;
   * имя призывателя;
   * регион;
   * лига;
   * League Points (LP).

5. Получение списка последних рейтинговых матчей каждого игрока.

6. Загрузка подробной информации по каждому матчу:

   * длительность;
   * режим игры;
   * версия клиента;
   * дата проведения;
   * статистика всех участников.

7. Формирование набора данных участников матчей (`participants.csv`).

8. Расчёт агрегированной статистики игроков:

   * количество матчей;
   * количество побед;
   * винрейт;
   * средние убийства, смерти и помощи;
   * сохранение результатов в `players_data.csv`.

9. Расчёт агрегированной статистики матчей:

   * регион;
   * лига;
   * длительность матча;
   * режим игры;
   * версия клиента;
   * дата проведения;
   * сохранение результатов в `matches_data.csv`.

Полученные данные используются в интерактивном дашборде для анализа статистики игроков, чемпионов и матчей с возможностью фильтрации по региону и рейтинговой лиге.

## Последовательность сбора данных

```
Регион (EUW / NA)
        ↓
Лига (Master / Grandmaster / Challenger)
        ↓
Получение списка игроков с наибольшим количеством LP
        ↓
Получение идентификаторов последних рейтинговых матчей (match_ids)
        ↓
Загрузка подробной информации о матчах
        ↓
Извлечение данных всех участников матчей
        ↓
Формирование итогового набора данных participants.csv
        ↓
Агрегация статистики игроков → players_data.csv
        ↓
Агрегация статистики матчей → matches_data.csv
        ↓
Использование данных в интерактивном дашборде
```


In [14]:
# Импорт библиотек для работы с API, обработки данных и отображения прогресса
import os
import time
import requests
import pandas as pd
from typing import List, Dict, Any
from dotenv import load_dotenv
from tqdm.auto import tqdm

In [27]:
# Загрузка API-ключа из файла .env
load_dotenv(override=True)
API_KEY = os.getenv("RIOT_API_KEY")

print(API_KEY)

RGAPI-f0d7d1ac-0314-48af-9ec4-2d9d7a6518ac


In [28]:
# Настройка регионов, лиг и заголовков для запросов к Riot API
REGIONS = {
    "euw": "euw1",
    "na": "na1"
}

REGIONAL_ROUTING = {
    "euw": "europe",
    "na": "americas"
}

LEAGUES = {
    "challenger": "challengerleagues",
    "grandmaster": "grandmasterleagues",
    "master": "masterleagues"
}

# Headers — это служебная информация, которую мы отправляем вместе с каждым запросом.
# Riot API проверяет наш ключ именно через заголовок X-Riot-Token.
headers = {
    "X-Riot-Token": API_KEY
}

In [29]:
# Проверка подключения к Riot Games API

# Формируем URL запроса к Challenger-лиге
url = (
    f"https://{REGIONS['euw']}.api.riotgames.com"
    f"/lol/league/v4/challengerleagues/by-queue/RANKED_SOLO_5x5"
)

# Отправляем GET-запрос
response = requests.get(url, headers=headers)

# Смотрим статус ответа
# 200 — успех, 401/403 — проблема с ключом, 429 — превышен лимит запросов
print("Статус ответа:", response.status_code)

Статус ответа: 200


In [30]:
# Функция отправки запросов к Riot API с учетом ограничений: 20 запросов в минуту, 100 за 2 минуты
def riot_get(url: str) -> Dict[str, Any]:

    # Заголовок с API-ключом для авторизации
    headers = {
        "X-Riot-Token": API_KEY
    }

    # Отправка GET-запроса
    response = requests.get(
        url,
        headers=headers
    )

    # Проверка превышения лимита запросов
    if response.status_code == 429:

        # Получение времени ожидания из ответа сервера
        retry_after = int(
            response.headers.get(
                "Retry-After",
                5
            )
        )

        # Вывод сообщения о задержке
        print(
            f"Лимит API. Ждем {retry_after} секунд"
        )

        # Ожидание перед повторным запросом
        time.sleep(retry_after)

        # Повторное выполнение запроса
        return riot_get(url)

    # Проверка успешности выполнения запроса
    response.raise_for_status()

    # Небольшая пауза между запросами
    time.sleep(0.06)

    # Возврат данных в формате JSON
    return response.json()

In [31]:
# Функция получения списка игроков из лиги
def get_league_players(region: str, league: str, queue: str = "RANKED_SOLO_5x5"):
    
    # Определяем эндпоинт лиги (master / grandmaster / challenger)
    league_endpoint = LEAGUES[league]

     # Формируем URL запроса к Riot API
    url = (
        f"https://{region}.api.riotgames.com"
        f"/lol/league/v4/{league_endpoint}/by-queue/{queue}"
    )

    # Отправляем запрос через универсальную функцию с обработкой лимитов
    data = riot_get(url)

    # Возвращаем список игроков (entries) из ответа API
    return data["entries"]

In [32]:
# Функция сбора игроков из всех регионов и лиг
def collect_players(players_per_group: int = 100):

    # Список для хранения всех игроков
    all_players = []

    # Проходим по всем регионам (euw, na)
    for region_name, region_code in REGIONS.items():

        # Проходим по всем лигам (master, grandmaster, challenger)
        for league_name, league_endpoint in LEAGUES.items():

             # Получаем игроков из конкретного региона и лиги
            players = get_league_players(
                region=region_code,
                league=league_name
            )

            # Сортируем игроков по количеству LP (по убыванию)
            players = sorted(
                players,
                key=lambda x: x["leaguePoints"],
                reverse=True
            )

            # Берем только топ-N игроков
            players = players[:players_per_group]

            # Добавляем информацию о регионе и лиге к каждому игроку
            for p in players:
                p["region"] = region_name
                p["league"] = league_name

            # Добавляем обработанных игроков в общий список
            all_players.extend(players)

    # Возвращаем итоговый список всех игроков
    return all_players

In [33]:
# Сбор выборки игроков: топ-300 по LP в каждой комбинации региона и лиги
players_df = pd.DataFrame(collect_players(300))
players_df

,puuid,leaguePoints,rank,wins,losses,veteran,inactive,freshBlood,hotStreak,region,league
0,t-B-3R7sKRM7TkpPLWjjaCj9JFZXkCP_J95KYmOVZLTkzM...,4070,I,493,411,True,False,False,False,euw,challenger
1,LUD_dY49Db5EwwzC7nN5e_nJcuFoInkZV_ToGx6mQZsXVb...,4028,I,435,288,True,False,False,False,euw,challenger
2,Zk3ILe0jgmGDSQMrR5SqcTDYLwJ3dXUee-KOhxnGi1OajE...,4010,I,809,676,True,False,False,True,euw,challenger
3,uPXhdGtCG056KTEiAK9AtSfapeJQjXWBrMAqAy1_kFFTwe...,3986,I,737,615,True,False,False,False,euw,challenger
4,ZayCP8hXM6tSMUUAywxbgNBz-rRLNmFc1b8RoeQplH7_c6...,3837,I,414,285,True,False,False,True,euw,challenger
...,...,...,...,...,...,...,...,...,...,...,...
1795,VG5okIDhVgA3iyqGufG1f6uIbY45YhqCv89ktJHxJje323...,731,I,359,298,False,False,False,False,na,master
1796,sJ-Gb3biMpMiTECKgqjKquFMhQXkVe6Xd_L1Cpipr5b5yG...,731,I,239,231,True,False,False,False,na,master
1797,vYKtFC5P4Czn3HIbnOKg9YV_ZFWcZ1zK_2V6gLDEMw36Yb...,730,I,210,176,False,False,False,False,na,master
1798,yXVcdQ7F43uOOac0MHgU5acp8Ss3IUdm7_Jw_-gXa9p8wZ...,730,I,161,152,True,False,False,True,na,master


In [34]:
# Проверка количества игроков в каждой комбинации регион + лига
players_df.groupby(["region", "league"]).size()

region  league     
euw     challenger     300
        grandmaster    300
        master         300
na      challenger     300
        grandmaster    300
        master         300
dtype: int64

In [35]:
# Сохранение объединенного датасета игроков в CSV-файл
players_df.to_csv("players_all_regions_leagues.csv", index=False)

In [36]:
# Функция получения списка матчей игрока по его PUUID
def get_match_ids(puuid: str, count: int, routing_region: str):
    url = (
        f"https://{routing_region}.api.riotgames.com"
        f"/lol/match/v5/matches/by-puuid/{puuid}/ids"
        f"?start=0&count={count}"
    )

    # Отправляем запрос через универсальную функцию Riot API
    return riot_get(url)

In [37]:
# Функция сбора match_id для всех игроков
def collect_match_ids(
    players_df: pd.DataFrame,
    matches_per_player: int = 5
):
    # Список для хранения результатов (match_id + игрок)
    result = []

    # Проходим по всем игрокам
    for _, player in tqdm(
        players_df.iterrows(),
        total=len(players_df),
        desc="Collecting match ids"
    ):

        # PUUID игрока
        puuid = player["puuid"]

        # Регион игрока (euw / na)
        region = player["region"]

        # Перевод региона в routing region (europe / americas)
        routing_region = REGIONAL_ROUTING[region]

        try:

            # Получение списка match_id для игрока
            match_ids = get_match_ids(
                puuid=puuid,
                count=matches_per_player,
                routing_region=routing_region
            )

            # Добавляем каждый match_id в итоговый список
            for match_id in match_ids:
                result.append({
                    "match_id": match_id,
                    "puuid": puuid,
                    "region": region,
                    "league": player["league"]
                })

        # Обработка ошибок API
        except Exception as e:

            # Лог ошибки с обрезанным puuid
            print(
                f"Error: {region} | "
                f"{player['league']} | "
                f"{puuid[:12]}... | {e}"
            )

    # Возвращаем DataFrame с match_id и метаданными
    return pd.DataFrame(result)

In [38]:
# Сбор match_id для всех игроков (по 2 матча на игрока)
match_ids_df = collect_match_ids(players_df, matches_per_player=2)

Error: euw | challenger | TmQ5KkGVq6zj... | HTTPSConnectionPool(host='europe.api.riotgames.com', port=443): Max retries exceeded with url: /lol/match/v5/matches/by-puuid/TmQ5KkGVq6zjFNKafUp7rLJKWg_WyndVbZep-ECvpdX97OIhB6g6J2dE7lj-IeaudigWGJvy2odqWA/ids?start=0&count=2 (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1028)')))
Лимит API. Ждем 47 секунд
Лимит API. Ждем 66 секунд
Лимит API. Ждем 65 секунд
Лимит API. Ждем 74 секунд
Лимит API. Ждем 72 секунд
Лимит API. Ждем 73 секунд
Лимит API. Ждем 59 секунд
Лимит API. Ждем 56 секунд
Лимит API. Ждем 58 секунд
Лимит API. Ждем 59 секунд
Лимит API. Ждем 58 секунд
Лимит API. Ждем 56 секунд
Лимит API. Ждем 57 секунд
Лимит API. Ждем 41 секунд


In [40]:
# Проверка количества собранных match_id
len(match_ids_df)

3598

In [42]:
# Распределение количества match_id по регионам и лигам
match_ids_df.groupby(["region", "league"]).size()

region  league     
euw     challenger     598
        grandmaster    600
        master         600
na      challenger     600
        grandmaster    600
        master         600
dtype: int64

In [77]:
# Просмотр таблицы match_id с привязкой к игрокам, регионам и лигам
match_ids_df

,match_id,puuid,region,league
0,EUW1_7877313650,0PjsQ010I4cNC8YOgXKqsI2yWvqq64riE_md839d9STqB2...,euw,challenger
1,EUW1_7877222111,0PjsQ010I4cNC8YOgXKqsI2yWvqq64riE_md839d9STqB2...,euw,challenger
2,EUW1_7877333021,hBBq3s_35ABN5zha2hzW3Xo5InrhRDrGiM5hvMaBNg4rYG...,euw,challenger
3,EUW1_7877300686,hBBq3s_35ABN5zha2hzW3Xo5InrhRDrGiM5hvMaBNg4rYG...,euw,challenger
4,EUW1_7877380914,LUD_dY49Db5EwwzC7nN5e_nJcuFoInkZV_ToGx6mQZsXVb...,euw,challenger
...,...,...,...,...
3595,NA1_5574900401,T7i1LpFAqt0WskkZKRwh4ZQK5ccfelsYoJA7YGhtRxlrvZ...,na,master
3596,NA1_5575443853,B9H0QmXsnVgVyNd01osGf7au8oHEdg4JLfWXDiq73onCLd...,na,master
3597,NA1_5575432696,B9H0QmXsnVgVyNd01osGf7au8oHEdg4JLfWXDiq73onCLd...,na,master
3598,NA1_5573643809,P7LbjyV3t7SnE4g9XaRjfBWDfmdvAoWvICSl87G1LDC7Bm...,na,master


In [43]:
# Сохранение всех match_id (по регионам и лигам) в CSV
match_ids_df.to_csv("match_ids_all_regions_leagues.csv", index=False)

In [44]:
# Получение данных матча по match_id (EUW1 / NA1)
def get_match(match_id: str):

    prefix = match_id.split("_")[0]

    # Определяем routing region только для используемых серверов
    if prefix == "NA1":
        routing = "americas"
    elif prefix == "EUW1":
        routing = "europe"
    else:
        raise ValueError(f"Unsupported region: {match_id}")

    # Формируем URL для Match-V5 API
    url = f"https://{routing}.api.riotgames.com/lol/match/v5/matches/{match_id}"

    return riot_get(url)

In [45]:
# Получение уникальных match_id для загрузки матчей
unique_match_ids = match_ids_df["match_id"].drop_duplicates().tolist()

print("Всего матчей для загрузки:", len(unique_match_ids))

Всего матчей для загрузки: 2558


In [47]:
# Загрузка сохраненного списка match_id из CSV-файла
match_ids_df = pd.read_csv("match_ids_all_regions_leagues.csv")

In [48]:
# Просмотр первых строк таблицы match_ids
match_ids_df.head()

,match_id,puuid,region,league
0,EUW1_7904163042,t-B-3R7sKRM7TkpPLWjjaCj9JFZXkCP_J95KYmOVZLTkzM...,euw,challenger
1,EUW1_7904147719,t-B-3R7sKRM7TkpPLWjjaCj9JFZXkCP_J95KYmOVZLTkzM...,euw,challenger
2,EUW1_7903341045,LUD_dY49Db5EwwzC7nN5e_nJcuFoInkZV_ToGx6mQZsXVb...,euw,challenger
3,EUW1_7903290156,LUD_dY49Db5EwwzC7nN5e_nJcuFoInkZV_ToGx6mQZsXVb...,euw,challenger
4,EUW1_7898064872,Zk3ILe0jgmGDSQMrR5SqcTDYLwJ3dXUee-KOhxnGi1OajE...,euw,challenger


In [49]:
# Функция загрузки информации о матчах по списку match_id
def collect_matches(match_ids_df):

    # Получаем уникальные match_id
    match_ids = match_ids_df["match_id"].drop_duplicates().tolist()

    # Вывод общего количества матчей для загрузки
    print("Всего матчей к загрузке:", len(match_ids))

    # Список для хранения данных матчей
    matches = []

     # Проходим по всем match_id
    for i, match_id in enumerate(tqdm(match_ids, desc="Загрузка матчей")):

        try:
            # Получение данных матча через Riot API
            match = get_match(match_id)

            # Добавляем матч в список, если он получен
            if match is not None:
                matches.append(match)

        except Exception as e:

            # Обработка ошибок загрузки конкретного матча
            print(f"[{i+1}/{len(match_ids)}] ошибка {match_id}: {e}")

        # Небольшая пауза для снижения нагрузки на API
        time.sleep(0.05)

        # Периодический вывод прогресса каждые 50 матчей
        if (i + 1) % 50 == 0:
            print(f"Загружено {i+1}/{len(match_ids)} матчей")

    # Возврат списка всех загруженных матчей
    return matches

In [50]:
# Загрузка всех матчей по списку match_id
matches = collect_matches(match_ids_df)

Всего матчей к загрузке: 2558


Загрузка матчей:   0%|          | 0/2558 [00:00<?, ?it/s]

Загружено 50/2558 матчей
Загружено 100/2558 матчей
Лимит API. Ждем 64 секунд
Загружено 150/2558 матчей
Загружено 200/2558 матчей
Лимит API. Ждем 64 секунд
Загружено 250/2558 матчей
Загружено 300/2558 матчей
Лимит API. Ждем 64 секунд
Загружено 350/2558 матчей
Загружено 400/2558 матчей
Лимит API. Ждем 63 секунд
Загружено 450/2558 матчей
Загружено 500/2558 матчей
Лимит API. Ждем 31 секунд
Загружено 550/2558 матчей
Загружено 600/2558 матчей
Лимит API. Ждем 57 секунд
[625/2558] ошибка EUW1_7904497434: HTTPSConnectionPool(host='europe.api.riotgames.com', port=443): Max retries exceeded with url: /lol/match/v5/matches/EUW1_7904497434 (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1028)')))
Загружено 650/2558 матчей
Загружено 700/2558 матчей
Загружено 750/2558 матчей
Загружено 800/2558 матчей
Загружено 850/2558 матчей
Загружено 900/2558 матчей
Загружено 950/2558 матчей
Загружено 1000/2558 матчей
Загружено 1050/2558 матчей
[

In [51]:
# Сохранение загруженных матчей в JSON-файл
import json

with open("matches.json", "w", encoding="utf-8") as f:
    json.dump(matches, f)

In [52]:
# Сохранение матчей в бинарный файл pickle
import pickle

with open("matches.pkl", "wb") as f:
    pickle.dump(matches, f)

In [53]:
# Функция преобразования матчей Riot API в таблицу участников
def create_participants_df(matches: List[Dict[str, Any]]) -> pd.DataFrame:
   
    # Список для хранения строк будущего DataFrame
    rows = []

    # Проходим по каждому матчу
    for match in matches:

        # ID матча из metadata
        match_id = match["metadata"]["matchId"]
        # Основная информация о матче
        info = match["info"]

        # Проходим по всем участникам матча
        for p in info["participants"]:

            # Формируем строку с данными игрока
            rows.append({
                "match_id": match_id,
                "puuid": p["puuid"],
                "summoner_name": p.get("summonerName"),
                "team_id": p.get("teamId"),

                "champion": p["championName"],
                "kills": p["kills"],
                "deaths": p["deaths"],
                "assists": p["assists"],

                "win": p["win"],

                # Дополнительно
                "gold": p.get("goldEarned"),
                "damage": p.get("totalDamageDealtToChampions"),
                "vision": p.get("visionScore")
            })

    # Преобразуем список словарей в DataFrame
    return pd.DataFrame(rows)

In [54]:
# Формирование таблицы участников матчей 
participants_df = create_participants_df(matches)

In [55]:
# Просмотр первых 10 строк таблицы участников матчей
participants_df.head(10)

,match_id,puuid,summoner_name,team_id,champion,kills,deaths,assists,win,gold,damage,vision
0,EUW1_7904163042,Oo_y27wOo7XnT820_1lTKE6pUT8FR7CeV-qmosqrBzs2-j...,,100,Pantheon,3,5,1,False,5306,9436,8
1,EUW1_7904163042,asjRzIRKaTcG0JF1a7ez5gz5OxR0kKv5mPI38ExTrAqkoU...,,100,Brand,2,3,1,False,5538,5826,20
2,EUW1_7904163042,tLAJes-ng7xZudy5dFKmiPqouutniIPVld59F_kfjIZLWa...,,100,Tristana,1,2,2,False,5491,10128,8
3,EUW1_7904163042,kkm_TgPTl_1uAX2mSay1N8AMlx4YHRwokY4wFgnRsmou1F...,,100,Varus,3,6,0,False,5623,9212,10
4,EUW1_7904163042,6ImWEYeY6hTAYoysf-pfv1iwtMpRyWXZ_dbXz4CO2B1q4N...,,100,Braum,0,3,4,False,3781,3965,34
5,EUW1_7904163042,SuJqnmua0PvKeaL8PcpDGt9mWQa_w0l0LB1dOxGCIReVLo...,,200,KSante,5,2,1,True,6613,8383,7
6,EUW1_7904163042,1ePB3mASeffK0Vr8X4hyu5avzEkhyy7Nn3PyIiElddn7oF...,,200,Nocturne,4,0,4,True,6667,5953,16
7,EUW1_7904163042,t-B-3R7sKRM7TkpPLWjjaCj9JFZXkCP_J95KYmOVZLTkzM...,,200,Akali,1,2,2,True,5620,10159,21
8,EUW1_7904163042,glczmhp0LsfJ7Cptiv_IQK0S3O_SeRc0aZ_S9LHoWGzPZw...,,200,Draven,8,0,1,True,10610,14743,10
9,EUW1_7904163042,TFk4MA2urGNzBqlB9pEwRMpBgNR4QUpXQvS3mkIjrl0LLc...,,200,Sylas,1,5,8,True,5380,9604,33


In [56]:
# Добавление информации о регионе и лиге к таблице участников матчей
participants_df = participants_df.merge(
    match_ids_df[["match_id", "region", "league"]].drop_duplicates(),
    on="match_id",
    how="left"
)

In [57]:
# Сохранение  таблицы участников матчей в CSV и Parquet
participants_df.to_csv("participants.csv", index=False)
participants_df.to_parquet("participants.parquet", index=False)

In [58]:
# Распределение участников матчей по регионам и лигам
participants_df.groupby(
    ["region", "league"]
).size()

region  league     
euw     challenger     4500
        grandmaster    5188
        master         5558
na      challenger     5256
        grandmaster    5540
        master         5886
dtype: int64

In [59]:
# Проверка размера таблицы
participants_df.shape

(31928, 14)

In [60]:
# Проверка количества уникальных матчей в таблице участников
participants_df["match_id"].nunique()

2547

In [61]:
# Функция создания таблицы матчей matches_df
def create_matches_df(matches):

    # Список для хранения строк DataFrame
    rows = []

    # Проходим по каждому матчу
    for m in matches:
        # Основная информация о матче
        info = m["info"]

        # Формируем строку с ключевыми характеристиками матча
        rows.append({
            "match_id": m["metadata"]["matchId"], # идентификатор матча
            "game_duration": info.get("gameDuration"), # длительность игры
            "game_mode": info.get("gameMode"),  # режим игры
            "game_version": info.get("gameVersion"), # версия
            "timestamp": info.get("gameCreation") # время создания матча
        })

    # Преобразование списка словарей в DataFrame
    return pd.DataFrame(rows)

In [62]:
# Формирование таблицы матче
matches_df = create_matches_df(matches)

In [63]:
# Преобразование timestamp в datetime и создание столбов дата и месяц
matches_df["datetime"] = pd.to_datetime(
    matches_df["timestamp"],
    unit="ms"
)
# Извлечение даты
matches_df["date"] = matches_df["datetime"].dt.date

# Извлечение месяца 
matches_df["month"] = matches_df["datetime"].dt.to_period("M")

In [64]:
# Просмотр первых 10 строк таблицы матчей
matches_df.head(10)

,match_id,game_duration,game_mode,game_version,timestamp,datetime,date,month
0,EUW1_7904163042,916,CLASSIC,16.13.791.5903,1782810577822,2026-06-30 09:09:37.822,2026-06-30,2026-06
1,EUW1_7904147719,1548,CLASSIC,16.13.791.5903,1782807749436,2026-06-30 08:22:29.436,2026-06-30,2026-06
2,EUW1_7903341045,1466,CLASSIC,16.13.791.5903,1782741585489,2026-06-29 13:59:45.489,2026-06-29,2026-06
3,EUW1_7903290156,1557,CLASSIC,16.13.791.5903,1782738217585,2026-06-29 13:03:37.585,2026-06-29,2026-06
4,EUW1_7898064872,1660,CLASSIC,16.13.789.3741,1782327936843,2026-06-24 19:05:36.843,2026-06-24,2026-06
5,EUW1_7897980110,1324,CLASSIC,16.13.789.3741,1782325142313,2026-06-24 18:19:02.313,2026-06-24,2026-06
6,EUW1_7904660886,1999,CLASSIC,16.13.791.5903,1782844582508,2026-06-30 18:36:22.508,2026-06-30,2026-06
7,EUW1_7904613897,1461,CLASSIC,16.13.791.5903,1782842314103,2026-06-30 17:58:34.103,2026-06-30,2026-06
8,EUW1_7904376410,1734,CLASSIC,16.13.791.5903,1782829699614,2026-06-30 14:28:19.614,2026-06-30,2026-06
9,EUW1_7904291191,1909,CLASSIC,16.13.791.5903,1782823489899,2026-06-30 12:44:49.899,2026-06-30,2026-06


In [65]:
# Сохранение таблицы матчей в CSV-файл
matches_df.to_csv("matches_data.csv", index=False)

In [66]:
# Загрузка сохранённой таблицы участников матчей из CSV
participants_df = pd.read_csv("participants.csv")

In [67]:
# Удаление дублей по игроку в рамках одного матча (уникальные пары match_id + puuid)
player_base = participants_df.drop_duplicates(subset=["match_id", "puuid"])

In [68]:
# Статистика игроков по puuid 
player_stats = player_base.groupby("puuid").agg(
    region=("region", "first"), # регион игрока (берём первое значение)
    league=("league", "first"), # лига игрока (Master / GM / Challenger)
    summoner_name=("summoner_name", "first"),  # ник игрока
    matches=("match_id", "nunique"), # количество уникальных матчей
    wins=("win", "sum"), # количество побед
    kills=("kills", "mean"), # среднее количество убийств за матч
    deaths=("deaths", "mean"), # среднее количество смертей
    assists=("assists", "mean") # среднее количество ассистов
).reset_index()

In [69]:
# Расчёт винрейта игроков

# Количество поражений
player_stats["losses"] = player_stats["matches"] - player_stats["wins"]

# Винрейт - доля побед от общего количества матчей
player_stats["winrate"] = (
    player_stats["wins"] / player_stats["matches"]
)

In [70]:
# Добавление информации о месяце матча к таблице участников
tmp = participants_df.merge(
    matches_df[["match_id", "month"]],
    on="match_id",
    how="left"
)

# Kоличество матчей на игрока
monthly = tmp.groupby("puuid").agg(
    matches_month=("match_id", "count")
).reset_index()

# Добавление активности в итоговую таблицу игроков
player_stats = player_stats.merge(monthly, on="puuid", how="left")

In [71]:
# Расчёт KDA (Kills + Assists / Deaths)
player_stats["kda"] = (
    (player_stats["kills"] + player_stats["assists"]) /
    player_stats["deaths"].replace(0, 1)
)

In [72]:
# Просмотр  таблицы статистики игроков
player_stats

,puuid,region,league,summoner_name,matches,wins,kills,deaths,assists,losses,winrate,matches_month,kda
0,--XJ2IG3dwIUxLZQocscjcusWi2IdMXUhsgL8GXsUPhPl6...,na,challenger,None,1,0,4.000000,6.0,6.000000,1,0.0,1,1.666667
1,--iU1KaH5tibCjBJe5YyWc7OtyNWmt681CfVkyhoLOFcfQ...,euw,challenger,None,1,1,13.000000,10.0,13.000000,0,1.0,1,2.600000
2,--u333E4cJ3dCxTWRlEqy6Nlu0-Vt2-yRKuOIj8I9wmSJp...,na,grandmaster,None,1,0,2.000000,3.0,2.000000,1,0.0,1,1.333333
3,-0gSL_BRBVJQcgdfPpuzIqQSHlo1XhMezgwMx0rvR-p1Ox...,na,master,None,1,1,0.000000,1.0,11.000000,0,1.0,1,11.000000
4,-1EIl85tomKDZ4ulVd935sNet4gWkcb1X5a8MMzLu_bimH...,na,master,None,1,0,2.000000,8.0,5.000000,1,0.0,1,0.875000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
12620,zyvACyrMq4ix-uEZX0LSThPmjC8aUZRlUjbBPynvOY1LHX...,na,master,None,1,1,0.000000,0.0,0.000000,0,1.0,1,0.000000
12621,zyy-BWUydf9fupgcDiK6G_8CVbSGr5QHWB6J4vkKm_pzHp...,na,master,None,1,1,11.000000,6.0,4.000000,0,1.0,1,2.500000
12622,zyz8dxCZNupBjsJi5ICXpgyc0nN4_kqF1JPJwWvauHzFpn...,na,challenger,None,3,3,8.333333,9.0,9.333333,0,1.0,3,1.962963
12623,zz2F2lsvY4RAla3sqkVvUmdxpdNf-o0NgGpOh_1HLO6Lzy...,euw,grandmaster,None,2,1,7.500000,11.5,6.000000,1,0.5,2,1.173913


In [73]:
# Сохранение итоговой таблицы статистики игроков в CSV файл
player_stats.to_csv("players_data.csv", index=False)

In [74]:
# Cтатистика по матчам и винрейту игроков
player_stats[["matches", "wins", "winrate"]].describe()

,matches,wins,winrate
count,12625.000000,12625.000000,12625.000000
mean,2.171723,1.086020,0.484784
std,2.512165,1.510593,0.435916
min,1.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000
50%,1.000000,1.000000,0.500000
75%,2.000000,1.000000,1.000000
max,28.000000,17.000000,1.000000


In [75]:
# Проверка корректности винрейта (не должен превышать 1.0)
player_stats[player_stats["winrate"] > 1]

,puuid,region,league,summoner_name,matches,wins,kills,deaths,assists,losses,winrate,matches_month,kda


In [76]:
# Проверка количества дублей match_id + puuid
player_base.duplicated(subset=["match_id", "puuid"]).sum()

np.int64(0)

In [77]:
# Агрегация статистики по чемпионам 
champion_stats = player_base.groupby(["champion"]).agg(
    matches=("champion", "size"), # количество сыгранных матчей на чемпиона
    wins=("win", "sum"), # количество побед на чемпионе
    kills=("kills", "mean"), # среднее количество убийств
    deaths=("deaths", "mean"), # среднее количество смертей
    assists=("assists", "mean") # среднее количество ассистов
).reset_index()

In [78]:
# Расчёт винрейта по чемпионам
champion_stats["losses"] = champion_stats["matches"] - champion_stats["wins"]

champion_stats["winrate"] = (
    champion_stats["wins"] / champion_stats["matches"]
)

In [79]:
# Расчёт KDA (Kills + Assists / Deaths) по чемпионам
champion_stats["kda"] = (
    (champion_stats["kills"] + champion_stats["assists"]) /
    champion_stats["deaths"].replace(0, 1)
)

In [80]:
# Просмотр итоговой таблицы статистики по чемпионам
champion_stats

,champion,matches,wins,kills,deaths,assists,losses,winrate,kda
0,Aatrox,193,90,5.943005,5.715026,6.575130,103,0.466321,2.190390
1,Ahri,246,108,6.650407,5.565041,8.121951,138,0.439024,2.654492
2,Akali,236,128,8.059322,5.338983,4.860169,108,0.542373,2.419841
3,Akshan,99,54,9.222222,6.010101,6.262626,45,0.545455,2.576471
4,Alistar,222,119,1.734234,6.279279,14.445946,103,0.536036,2.576758
...,...,...,...,...,...,...,...,...,...
168,Zeri,178,89,7.449438,6.353933,6.977528,89,0.500000,2.270557
169,Ziggs,91,44,5.879121,5.989011,9.098901,47,0.483516,2.500917
170,Zilean,117,63,2.145299,4.974359,13.931624,54,0.538462,3.231959
171,Zoe,158,76,5.544304,5.360759,9.082278,82,0.481013,2.728453


In [81]:
# Сохранение итоговой статистики по чемпионам в CSV файл
champion_stats.to_csv("champions_data.csv", index=False)

In [82]:
# Добавление LP к таблице игроков через merge по puuid

# Загрузка таблицы игроков с регионами и лигами 
players_rank = pd.read_csv("players_all_regions_leagues.csv")

# Загрузка итоговой статистики игроков
players_data = pd.read_csv("players_data.csv")

# Добавление LP (league points) к таблице игроков
players_data = players_data.merge(
    players_rank[
        ["puuid", "leaguePoints"] # берем только  столбцы puuid и LP
    ],
    on="puuid",
    how="left"
)

In [83]:
# Просмотр итоговой таблицы игроков с добавленным LP
players_data

,puuid,region,league,summoner_name,matches,wins,kills,deaths,assists,losses,winrate,matches_month,kda,leaguePoints
0,--XJ2IG3dwIUxLZQocscjcusWi2IdMXUhsgL8GXsUPhPl6...,na,challenger,NaN,1,0,4.000000,6.0,6.000000,1,0.0,1,1.666667,NaN
1,--iU1KaH5tibCjBJe5YyWc7OtyNWmt681CfVkyhoLOFcfQ...,euw,challenger,NaN,1,1,13.000000,10.0,13.000000,0,1.0,1,2.600000,NaN
2,--u333E4cJ3dCxTWRlEqy6Nlu0-Vt2-yRKuOIj8I9wmSJp...,na,grandmaster,NaN,1,0,2.000000,3.0,2.000000,1,0.0,1,1.333333,NaN
3,-0gSL_BRBVJQcgdfPpuzIqQSHlo1XhMezgwMx0rvR-p1Ox...,na,master,NaN,1,1,0.000000,1.0,11.000000,0,1.0,1,11.000000,NaN
4,-1EIl85tomKDZ4ulVd935sNet4gWkcb1X5a8MMzLu_bimH...,na,master,NaN,1,0,2.000000,8.0,5.000000,1,0.0,1,0.875000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12620,zyvACyrMq4ix-uEZX0LSThPmjC8aUZRlUjbBPynvOY1LHX...,na,master,NaN,1,1,0.000000,0.0,0.000000,0,1.0,1,0.000000,NaN
12621,zyy-BWUydf9fupgcDiK6G_8CVbSGr5QHWB6J4vkKm_pzHp...,na,master,NaN,1,1,11.000000,6.0,4.000000,0,1.0,1,2.500000,NaN
12622,zyz8dxCZNupBjsJi5ICXpgyc0nN4_kqF1JPJwWvauHzFpn...,na,challenger,NaN,3,3,8.333333,9.0,9.333333,0,1.0,3,1.962963,1320.0
12623,zz2F2lsvY4RAla3sqkVvUmdxpdNf-o0NgGpOh_1HLO6Lzy...,euw,grandmaster,NaN,2,1,7.500000,11.5,6.000000,1,0.5,2,1.173913,NaN


In [84]:
# Сохранение обновлённой таблицы игроков с LP в CSV файл
players_data.to_csv("players_data.csv", index=False)

In [85]:
# Добавление информации о регионе и лиге к матчам

# Загрузка таблицы match_id с регионами и лигами
match_ids = pd.read_csv("match_ids_all_regions_leagues.csv")

# Загрузка агрегированной таблицы матчей
matches_data = pd.read_csv("matches_data.csv")

# Добавление информации о регионе и лиге к матчам 
matches_data = matches_data.merge(
    match_ids[["match_id", "region", "league"]],
    on="match_id",
    how="left"
)

In [86]:
# Просмотр итоговой таблицы матчей с добавленными информации о регионе и лиге
matches_data

,match_id,game_duration,game_mode,game_version,timestamp,datetime,date,month,region,league
0,EUW1_7904163042,916,CLASSIC,16.13.791.5903,1782810577822,2026-06-30 09:09:37.822,2026-06-30,2026-06,euw,challenger
1,EUW1_7904163042,916,CLASSIC,16.13.791.5903,1782810577822,2026-06-30 09:09:37.822,2026-06-30,2026-06,euw,challenger
2,EUW1_7904147719,1548,CLASSIC,16.13.791.5903,1782807749436,2026-06-30 08:22:29.436,2026-06-30,2026-06,euw,challenger
3,EUW1_7903341045,1466,CLASSIC,16.13.791.5903,1782741585489,2026-06-29 13:59:45.489,2026-06-29,2026-06,euw,challenger
4,EUW1_7903290156,1557,CLASSIC,16.13.791.5903,1782738217585,2026-06-29 13:03:37.585,2026-06-29,2026-06,euw,challenger
...,...,...,...,...,...,...,...,...,...,...
3581,NA1_5590010052,1226,CLASSIC,16.13.791.5903,1782537087455,2026-06-27 05:11:27.455,2026-06-27,2026-06,na,master
3582,NA1_5592213875,1857,CLASSIC,16.13.791.5903,1782813883250,2026-06-30 10:04:43.250,2026-06-30,2026-06,na,master
3583,NA1_5592205662,1540,CLASSIC,16.13.791.5903,1782811892618,2026-06-30 09:31:32.618,2026-06-30,2026-06,na,master
3584,NA1_5592221082,2146,CLASSIC,16.13.791.5903,1782817553814,2026-06-30 11:05:53.814,2026-06-30,2026-06,na,master


In [87]:
# Сохранение обновлённой таблицы матчей в CSV файл
matches_data.to_csv("matches_data.csv", index=False)